# Trabajo Final Evaluativo — APIs y Web Crawling / Web Scraping
**Alumno:** Aramayo Pedro Mauricio Leandro — 3.er año, Ciencias de Datos e IA

Un solo cuaderno con los 10 ejercicios del PDF (se conserva la numeración original). El orden de trabajo fue 3 → 4 → 9 → 10 y después el resto.
Tema elegido para los puntos 3, 9 y 10: **inflación** (tiene muchas noticias en diarios argentinos todos los días).

**Cómo usarlo:** ejecutar las celdas en orden (*Entorno de ejecución → Ejecutar todas*). Las recolecciones lentas (descarga de textos, X con navegador) quedan guardadas en la carpeta `datos/` del repositorio; cada una tiene una variable `RECOLECTAR_... = False` que se puede poner en `True` para repetirla en vivo.

| Etiqueta | Significado |
|---|---|
| **API** | Se consulta un servicio que devuelve datos estructurados (RSS, OAI-PMH, SPARQL). |
| **Crawling** | El programa sigue enlaces/páginas solo (paginación, "ver más", tokens). |
| **Scraping** | Se extraen datos del HTML de una página pensada para personas. |

In [ ]:
# 0. Preparación: descarga los datos del repositorio (en Colab) e instala lo que falta.
import os, sys
REPO = 'https://github.com/LeandroAram/tf-apis-web-scraping.git'
RAMA = 'devin/1795598835-notebook-tf'   # si la rama ya se unió a main, poner 'main'
if not os.path.exists('datos'):
    !git clone -q {REPO} repo_tf
    %cd repo_tf
    !git checkout -q {RAMA} 2>/dev/null || true
!pip install -q pypdf python-docx
print('Carpeta de trabajo:', os.getcwd())
print('Archivos de datos:', sorted(os.listdir('datos')))

In [ ]:
import re, json, time, unicodedata, urllib.parse, xml.etree.ElementTree as ET
from collections import Counter
from datetime import datetime
import requests, pandas as pd, matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from IPython.display import display, HTML, Image
import nltk
nltk.download('stopwords', quiet=True); nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords

pd.set_option('display.max_colwidth', 120)
UA = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36',
      'Accept-Language': 'es-AR,es;q=0.9'}
ESTADO = {}   # acá cada ejercicio anota si quedó completo o no

def sin_tildes(texto):
    texto = unicodedata.normalize('NFD', str(texto).lower())
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

VACIAS = set(sin_tildes(p) for p in stopwords.words('spanish')) | {
    'ademas', 'segun', 'tambien', 'solo', 'mas', 'asi', 'dijo', 'aseguro', 'https', 'http', 'www', 'com', 'ser', 'hace',
    'puede', 'anos', 'ano', 'dos', 'tres', 'mientras', 'luego', 'tras', 'hoy', 'ver', 'mismo', 'sido', 'cada', 'parte', 'ahora'}

def palabras(texto):
    """Pasa a minúsculas, saca tildes y palabras vacías (de, la, que...)."""
    return [p for p in re.findall(r'[a-zñ]{3,}', sin_tildes(texto)) if p not in VACIAS]

import html as html_lib
def link(url, texto):
    return f'<a href="{html_lib.escape(url)}" target="_blank">{html_lib.escape(texto)}</a>'

---
## Ejercicio 1 — Seguimiento de precios en Mercado Libre
**Objetivo:** seguir 15 artículos (nombre, precio, imagen), guardar el precio con fecha y avisar si cambió respecto de la ejecución anterior.

**Fuente:** `listado.mercadolibre.com.ar` (**scraping**) y `api.mercadolibre.com` (**API**).

In [ ]:
BUSQUEDA_ML = 'auriculares bluetooth'
URL_ML = 'https://listado.mercadolibre.com.ar/' + BUSQUEDA_ML.replace(' ', '-')
API_ML = 'https://api.mercadolibre.com/sites/MLA/search?q=' + urllib.parse.quote(BUSQUEDA_ML)
HISTORIAL_ML = 'datos/ej1_historial_mercadolibre.csv'

def articulos_mercadolibre(cantidad=15):
    r = requests.get(URL_ML, headers=UA, timeout=30)
    if 'captcha' in r.url or 'verification' in r.url:
        raise PermissionError('Mercado Libre redirigió a una verificación de seguridad: ' + r.url.split('?')[0])
    sopa = BeautifulSoup(r.text, 'html.parser')
    articulos = []
    for item in sopa.select('li.ui-search-layout__item')[:cantidad]:
        enlace = item.select_one('a[href*="MLA"]')
        precio = item.select_one('.andes-money-amount__fraction')
        imagen = item.select_one('img')
        articulos.append({
            'id': re.search(r'MLA-?\d+', enlace['href']).group().replace('-', ''),
            'nombre': enlace.get_text(strip=True),
            'precio': int(precio.get_text(strip=True).replace('.', '')),
            'imagen': imagen.get('data-src') or imagen.get('src'),
            'url': enlace['href'].split('#')[0],
            'fecha': datetime.now().strftime('%Y-%m-%d %H:%M')})
    return articulos

print('Prueba con la API pública:', requests.get(API_ML, headers=UA, timeout=30).status_code, '(403 = acceso denegado)')
try:
    actuales = pd.DataFrame(articulos_mercadolibre())
except PermissionError as error:
    actuales = pd.DataFrame()
    print('BLOQUEADO ->', error)

In [ ]:
# Historial + alertas: solo se ejecuta si se pudieron leer artículos reales.
if len(actuales):
    anterior = pd.read_csv(HISTORIAL_ML) if os.path.exists(HISTORIAL_ML) else pd.DataFrame(columns=actuales.columns)
    ultimo = anterior.sort_values('fecha').groupby('id').last()
    for _, a in actuales.iterrows():
        if a['id'] in ultimo.index and ultimo.loc[a['id'], 'precio'] != a['precio']:
            print(f"ALERTA: {a['nombre'][:60]} pasó de ${ultimo.loc[a['id'], 'precio']:,} a ${a['precio']:,}")
    if anterior.empty:
        print('Primera ejecución: se guardan estos precios como referencia.')
    actuales.to_csv(HISTORIAL_ML, mode='a', header=not os.path.exists(HISTORIAL_ML), index=False)
    display(HTML(''.join(f"<div style='display:inline-block;width:150px;margin:4px'><img src='{a.imagen}' width=120><br>{a.nombre[:50]}<br><b>${a.precio:,}</b></div>" for a in actuales.itertuples())))
    ESTADO[1] = 'Completo' if len(actuales) >= 15 else f'No completo ({len(actuales)} artículos)'
else:
    ESTADO[1] = 'No completo: Mercado Libre bloquea la recolección (captcha en la web y 403 en la API)'
print(ESTADO[1])

**Qué pude verificar:** desde la máquina de prueba (y probablemente también desde Colab, que usa IPs de centros de datos) Mercado Libre redirige la búsqueda a `captcha/wall/web-risk-verification` y la API responde **403**. No se cargaron precios inventados: el historial y las alertas quedan programados y funcionan apenas el sitio devuelve la página real (por ejemplo, desde una PC hogareña). **Estado: no completo.**

---
## Ejercicio 2 — Avisos clasificados en tres sitios
**Objetivo:** comparar el precio de una misma categoría (**Volkswagen Gol Trend usado**) en tres sitios y avisar cuando un mismo aviso cambia de precio.

**Fuentes (scraping):** RosarioGarage (HTML), Autocosmos (microdatos `itemprop` del HTML) y Kavak (datos JSON dentro de la página).

In [ ]:
def descargar(url, intentos=3):
    """Descarga una página; reintenta porque algunos sitios responden lento."""
    for i in range(intentos):
        try:
            r = requests.get(url, headers=UA, timeout=90)
            if r.status_code == 200 and len(r.content) > 20000:
                return r
        except requests.RequestException:
            pass
    raise RuntimeError(f'No se pudo descargar {url}')

def a_numero(texto):
    """'$ 11.500.000' -> 11500000 ; 'Consultar' -> None"""
    digitos = re.sub(r'\D', '', texto or '')
    return int(digitos) if digitos else None

def avisos_rosariogarage():
    url = 'https://www.rosariogarage.com/Autos/Volkswagen/Gol-Trend'
    sopa = BeautifulSoup(descargar(url).content, 'html.parser')
    avisos = []
    for caja in sopa.select('div[data-rel]'):
        titulo = next((a for a in caja.select('a[href*=showProduct]') if a.get_text(strip=True)), None)
        texto = caja.get_text(' ', strip=True)
        m = re.search(r'(u\$s|\$)\s*([\d\.]+)', texto)
        anio = re.search(r'\b(19|20)\d{2}\b', texto)
        avisos.append({'fuente': 'RosarioGarage', 'id_aviso': 'RG-' + caja['data-rel'],
                       'aviso': ' '.join(titulo.get_text(' ', strip=True).split()) if titulo else texto[:80],
                       'anio': int(anio.group()) if anio else None,
                       'moneda': ('USD' if m.group(1) == 'u$s' else 'ARS') if m else None,
                       'precio': a_numero(m.group(2)) if m else None,
                       'url': titulo.get('href') if titulo else url})
    return avisos

def avisos_autocosmos():
    url = 'https://www.autocosmos.com.ar/auto/usado/volkswagen/gol-trend'
    sopa = BeautifulSoup(descargar(url).content, 'html.parser')
    avisos = []
    for art in sopa.select('article.listing-card'):
        enlace = art.select_one('a[itemprop=url]')
        precio = art.select_one('[itemprop=price]')
        moneda = art.select_one('[itemprop=priceCurrency]')
        anio = art.select_one('[itemprop=modelDate]')
        avisos.append({'fuente': 'Autocosmos', 'id_aviso': 'AC-' + enlace['href'].rstrip('/').split('/')[-1],
                       'aviso': 'Volkswagen Gol Trend ' + art.select_one('.listing-card__version').get_text(strip=True),
                       'anio': int(anio.get_text(strip=True)) if anio else None,
                       'moneda': moneda['content'] if moneda else None,
                       'precio': a_numero(precio.get('content')) if precio else None,
                       'url': 'https://www.autocosmos.com.ar' + enlace['href']})
    return avisos

def avisos_kavak():
    url = 'https://www.kavak.com/ar/usados/volkswagen/gol-trend'
    html = descargar(url).text.replace('\\"', '"')
    patron = r'\{"id":"(\d+)","url":"([^"]+)","image":"[^"]*".*?"subtitle":"([^"]+)".*?"mainPrice":"([^"]+)"'
    avisos = []
    for id_, enlace, subtitulo, precio in re.findall(patron, html):
        partes = subtitulo.split(' • ')   # "2017 • 85.000 km • 1.6 Trendline"
        avisos.append({'fuente': 'Kavak', 'id_aviso': 'KV-' + id_, 'aviso': 'Volkswagen Gol Trend ' + partes[-1],
                       'anio': int(partes[0]), 'moneda': 'ARS', 'precio': a_numero(precio), 'url': enlace})
    return avisos

In [ ]:
HISTORIAL_AVISOS = 'datos/ej2_historial_avisos.csv'
ahora = datetime.now().strftime('%Y-%m-%d %H:%M')
todos = []
for sitio in [avisos_rosariogarage, avisos_autocosmos, avisos_kavak]:
    try:
        nuevos = sitio(); todos += nuevos
        print(f'{sitio.__name__}: {len(nuevos)} avisos')
    except Exception as error:
        print(f'{sitio.__name__}: ERROR {error}')
hoy = pd.DataFrame(todos).drop_duplicates('id_aviso'); hoy['fecha_hora'] = ahora   # un sitio puede repetir un aviso destacado
display(hoy[['fuente', 'aviso', 'anio', 'moneda', 'precio']].groupby('fuente').head(3))

In [ ]:
# Comparación entre sitios (solo avisos con precio en pesos; "Consultar" queda sin precio)
pesos = hoy[hoy.moneda == 'ARS']
tabla = pesos.groupby('fuente').precio.agg(avisos='count', minimo='min', mediana='median', maximo='max').astype(int)
display(tabla.style.format('{:,.0f}'))
por_anio = pesos[pesos.anio.between(2012, 2018)].pivot_table(index='anio', columns='fuente', values='precio', aggfunc='median')
display((por_anio / 1e6).round(1).style.set_caption('Mediana de precio por año del auto (millones de $)'))
print('Avisos sin precio publicado ("Consultar"):', hoy.precio.isna().sum(), '| en dólares:', (hoy.moneda == 'USD').sum())

In [ ]:
# Alertas: comparo cada aviso con su precio en la ejecución anterior guardada en el historial
historial = pd.read_csv(HISTORIAL_AVISOS) if os.path.exists(HISTORIAL_AVISOS) else pd.DataFrame(columns=hoy.columns)
previas = historial[historial.fecha_hora < ahora]
cambios = []
if previas.empty:
    print('Primera ejecución: el historial queda como referencia para la próxima comparación.')
else:
    ultima = previas[previas.fecha_hora == previas.fecha_hora.max()].set_index('id_aviso')
    print('Comparando contra la ejecución del', previas.fecha_hora.max())
    cruce = hoy.join(ultima[['precio', 'moneda']], on='id_aviso', rsuffix='_antes')
    cambios = cruce[cruce.precio_antes.notna() & cruce.precio.notna() & (cruce.precio != cruce.precio_antes)]
    for c in cambios.itertuples():
        print(f'ALERTA [{c.fuente}] {c.aviso[:50]}: {c.moneda} {c.precio_antes:,.0f} -> {c.precio:,.0f}')
    print('Avisos con cambio de precio:', len(cambios))
    print('Avisos nuevos:', (~hoy.id_aviso.isin(ultima.index)).sum(), '| avisos que ya no están:', (~ultima.index.isin(hoy.id_aviso)).sum())
hoy.to_csv(HISTORIAL_AVISOS, mode='a', header=not os.path.exists(HISTORIAL_AVISOS), index=False)
ESTADO[2] = (f'Completo: 3 sitios comparados; {len(cambios)} cambio(s) de precio respecto de la ejecución anterior'
            if hoy.fuente.nunique() >= 3 and not previas.empty else 'No completo: falta una segunda ejecución para comparar')
print('Ejecuciones guardadas en el historial:', pd.read_csv(HISTORIAL_AVISOS).fecha_hora.nunique(), '| Estado:', ESTADO[2])

---
## Ejercicio 3 — Búsqueda temática en diez diarios digitales
**Objetivo:** buscar "inflación" en diez diarios argentinos, contar cuántas noticias aparecen en cada uno, mostrar titulares con enlaces y sacar una conclusión a partir de los textos.

**Fuentes:** Google News RSS con el filtro `site:` de cada diario (**API**: devuelve XML), resolución del enlace real de cada noticia y descarga del cuerpo de la nota desde cada diario (**scraping**: JSON-LD `articleBody` o párrafos `<p>`). Recorrer resultados → seguir el enlace → abrir la nota es el **crawling**.

In [ ]:
TEMA = 'inflación'
DIAS = 7
DIARIOS = {'Clarín': 'clarin.com', 'La Nación': 'lanacion.com.ar', 'Infobae': 'infobae.com', 'Página/12': 'pagina12.com.ar',
           'Perfil': 'perfil.com', 'Ámbito': 'ambito.com', 'El Cronista': 'cronista.com', 'TN': 'tn.com.ar',
           'El Destape': 'eldestapeweb.com', 'iProfesional': 'iprofesional.com'}

def buscar_en_diario(consulta, dominio, dias=DIAS):
    """Consulta el RSS de Google News limitado a un diario y a los últimos días."""
    url = 'https://news.google.com/rss/search?' + urllib.parse.urlencode(
        {'q': f'{consulta} site:{dominio} when:{dias}d', 'hl': 'es-419', 'gl': 'AR', 'ceid': 'AR:es-419'})
    raiz = ET.fromstring(requests.get(url, headers=UA, timeout=30).content)
    return [{'titulo': re.sub(r'\s+-\s+[^-]+$', '', it.findtext('title')), 'link_google': it.findtext('link'),
             'fecha': it.findtext('pubDate')} for it in raiz.iter('item')]

filas = []
for nombre, dominio in DIARIOS.items():
    for i, r in enumerate(buscar_en_diario(TEMA, dominio)):
        filas.append({'diario': nombre, 'posicion': i + 1, **r})
busqueda = pd.DataFrame(filas)
conteo = busqueda.diario.value_counts().reindex(DIARIOS).rename('noticias encontradas')
display(conteo.to_frame().T)
conteo.plot.barh(title=f'Noticias con "{TEMA}" en los últimos {DIAS} días (Google News devuelve hasta 100 por consulta)', figsize=(8, 4)); plt.show()
print('Total de resultados:', len(busqueda))

La función que sigue convierte el enlace intermedio de Google News en la URL real del diario y descarga el texto. Como bajar más de 100 notas tarda unos minutos, si ya existen se usan las notas guardadas en `datos/noticias_<tema>.csv` (para inflación, `datos/noticias_inflacion.csv`) (recolectadas con este mismo código).

In [ ]:
def url_real(link_google):
    """Google News entrega un enlace intermedio; este pedido devuelve la dirección real de la nota."""
    gid = link_google.split('/articles/')[1].split('?')[0]
    html = requests.get(f'https://news.google.com/rss/articles/{gid}', headers=UA, timeout=30).text
    firma = re.search(r'data-n-a-sg="([^"]+)"', html).group(1)
    ts = re.search(r'data-n-a-ts="([^"]+)"', html).group(1)
    pedido = ['Fbv4je', f'[["garturlreq",[["X","X",["X","X"],null,null,1,1,"US:en",null,1,null,null,null,null,null,0,1],"X","X",1,[1,1,1],1,1,null,0,0,null,0],"{gid}",{ts},"{firma}"]']
    r = requests.post('https://news.google.com/_/DotsSplashUi/data/batchexecute',
                      headers={**UA, 'Content-Type': 'application/x-www-form-urlencoded;charset=UTF-8'},
                      data='f.req=' + urllib.parse.quote(json.dumps([[pedido]])), timeout=30)
    return json.loads(json.loads(r.text.split('\n\n')[1])[0][2])[1]

def texto_noticia(url):
    """Saca el cuerpo de la nota: primero del JSON-LD (articleBody), si no de los párrafos <p>."""
    html = requests.get(url, headers=UA, timeout=30).content
    sopa = BeautifulSoup(html, 'html.parser')
    m = re.search(rb'Fusion\.globalContent=(\{.*?\});Fusion\.', html, re.S)   # sitios Arc, p. ej. Página/12
    if m:
        try:
            t = ' '.join(BeautifulSoup(e.get('content', ''), 'html.parser').get_text(' ')
                         for e in json.loads(m.group(1)).get('content_elements', []) if e.get('type') == 'text')
            if len(t) > 400: return t.strip()
        except Exception: pass
    for bloque in sopa.find_all('script', type='application/ld+json'):
        try:
            datos = json.loads(bloque.string or '')
            for x in (datos if isinstance(datos, list) else datos.get('@graph', [datos])):
                if isinstance(x, dict) and len(x.get('articleBody') or '') > 300:
                    return x['articleBody'].strip()
        except Exception: pass
    texto = ''
    for zona in [sopa.find('article'), sopa]:
        if zona is not None:
            texto = ' '.join(p.get_text(' ', strip=True) for p in zona.find_all('p') if len(p.get_text(strip=True)) > 60)
            if len(texto) > 400: break
    return texto

ARCHIVO_NOTAS = 'datos/noticias_' + re.sub(r'\W+', '_', sin_tildes(TEMA)) + '.csv'
# Si el tema cambió y no hay notas guardadas, se descargan (~13 por diario, tarda unos 5 minutos)
RECOLECTAR_TEXTOS = not os.path.exists(ARCHIVO_NOTAS)
if RECOLECTAR_TEXTOS:
    notas = []
    for nombre, g in busqueda.groupby('diario', sort=False):
        for f in g.itertuples():
            if sum(n['diario'] == nombre for n in notas) >= 13: break
            try:
                u = url_real(f.link_google); t = texto_noticia(u)
                if len(t) >= 400:
                    notas.append({'diario': nombre, 'titulo': f.titulo, 'fecha': f.fecha, 'url': u, 'texto': t, 'busqueda': TEMA})
            except Exception as e:
                print('No se pudo leer una nota de', nombre, '-', str(e)[:60])
    pd.DataFrame(notas).drop_duplicates('url').to_csv(ARCHIVO_NOTAS, index=False)

noticias = pd.read_csv(ARCHIVO_NOTAS)
print(f'Notas con texto completo: {len(noticias)} (de {noticias.diario.nunique()} diarios)')
display(noticias.diario.value_counts().to_frame('notas con texto').T)

In [ ]:
# Algunos titulares con su enlace (dos por diario)
muestra = noticias.groupby('diario').head(2)
display(HTML('<ul>' + ''.join(f'<li><b>{f.diario}</b>: {link(f.url, f.titulo)}</li>' for f in muestra.itertuples()) + '</ul>'))

In [ ]:
# ¿De qué hablan los textos? Palabras y pares de palabras más frecuentes
todas = Counter(); pares = Counter()
for t in noticias.texto:
    p = palabras(t); todas.update(p); pares.update(zip(p, p[1:]))
display(pd.DataFrame({'palabra': [w for w, _ in todas.most_common(15)], 'veces': [n for _, n in todas.most_common(15)],
                      'par de palabras': [' '.join(w) for w, _ in pares.most_common(15)], 'veces ': [n for _, n in pares.most_common(15)]}))
TEMAS_ASOCIADOS = {'salarios': 'salari', 'tarifas': 'tarifa', 'alimentos': 'aliment', 'dólar': 'dolar', 'tasas de interés': 'tasa',
                   'Banco Central': 'banco central|bcra', 'INDEC / IPC': 'indec|ipc', 'recesión / actividad': 'recesion|actividad'}
asociados = pd.Series({k: noticias.texto.map(sin_tildes).str.contains(v).mean() * 100 for k, v in TEMAS_ASOCIADOS.items()}).sort_values()
asociados.plot.barh(title='% de notas sobre inflación que mencionan cada tema', figsize=(7, 3.5)); plt.show()
ESTADO[3] = 'Completo' if busqueda.diario.nunique() == 10 and noticias.diario.nunique() == 10 else 'No completo'
print('Estado:', ESTADO[3])

**Conclusión (sobre las 130 notas descargadas):** los diez diarios publican sobre inflación todos los días (casi todos llegan al tope de 100 resultados por semana). En los textos, inflación aparece casi siempre junto con **tasas de interés** (≈46 % de las notas), el **dólar** (≈42 %) y el dato del **INDEC/IPC** (≈38 %); un tercio habla de **salarios** y del Banco Central. Es decir, los diarios tratan la inflación más como un tema financiero y de política económica que como un tema de góndola (alimentos aparece en menos del 20 %). **Estado: completo.**

---
## Ejercicio 4 — Repositorio institucional con más de 1.000 registros
**Objetivo:** recuperar publicaciones de un repositorio grande y mostrar cómo se distribuyen por año, tipo y tema.

**Fuente:** SEDICI, repositorio de la Universidad Nacional de La Plata, mediante su servicio **OAI-PMH (API)**. Las respuestas vienen de a 100 registros con un `resumptionToken` para pedir la página siguiente (**crawling** de páginas).

In [ ]:
OAI = 'https://sedici.unlp.edu.ar/oai/request'
CONJUNTO = 'com_10915_36'   # Facultad de Informática

def tamanio(parametros):
    raiz = ET.fromstring(requests.get(OAI, params={'verb': 'ListIdentifiers', 'metadataPrefix': 'oai_dc', **parametros}, timeout=120).content)
    return int(raiz.find('.//{http://www.openarchives.org/OAI/2.0/}resumptionToken').get('completeListSize'))

print(f'Evidencia del tamaño (dato que informa el propio repositorio):')
print(f'  Registros en todo SEDICI:         {tamanio({}):,}')
print(f'  Registros en Facultad de Informática: {tamanio({"set": CONJUNTO}):,}')

In [ ]:
NS = {'oai': 'http://www.openarchives.org/OAI/2.0/', 'dc': 'http://purl.org/dc/elements/1.1/', 'oai_dc': 'http://www.openarchives.org/OAI/2.0/oai_dc/'}

def cosechar(conjunto):
    filas, params = [], {'verb': 'ListRecords', 'metadataPrefix': 'oai_dc', 'set': conjunto}
    while True:
        raiz = ET.fromstring(requests.get(OAI, params=params, timeout=120).content)
        for rec in raiz.iter('{%s}record' % NS['oai']):
            md_ = rec.find('.//oai_dc:dc', NS)
            if md_ is None: continue
            campo = lambda t: [e.text for e in md_.findall('dc:' + t, NS) if e.text]
            filas.append({'id': rec.findtext('.//oai:identifier', namespaces=NS), 'titulo': (campo('title') or [''])[0],
                          'fecha': (campo('date') or [''])[0], 'tipo': '; '.join(campo('type')), 'idioma': '; '.join(campo('language')),
                          'temas': '; '.join(campo('subject')), 'resumen': (campo('description') or [''])[0][:1500]})
        token = raiz.find('.//oai:resumptionToken', NS)
        if token is None or not token.text: return pd.DataFrame(filas)
        params = {'verb': 'ListRecords', 'resumptionToken': token.text}

RECOLECTAR_SEDICI = False   # True = volver a cosechar los registros (tarda 1-2 minutos)
if RECOLECTAR_SEDICI:
    cosechar(CONJUNTO).to_csv('datos/ej4_sedici_informatica.csv', index=False)
sedici = pd.read_csv('datos/ej4_sedici_informatica.csv')
print('Publicaciones recuperadas y analizadas:', len(sedici))
display(sedici[['titulo', 'fecha', 'tipo']].head())

In [ ]:
sedici['anio'] = pd.to_numeric(sedici.fecha.str[:4], errors='coerce')
fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
sedici.anio.value_counts().sort_index().plot.bar(ax=ejes[0], title='Publicaciones por año')
sedici.tipo.str.split(';').str[0].value_counts().head(8).sort_values().plot.barh(ax=ejes[1], title='Tipo de publicación')
plt.tight_layout(); plt.show()
temas = sedici.temas.dropna().str.split(';').explode().str.strip().str.capitalize()
temas = temas[(temas != '') & (temas != 'Ciencias informáticas')]
display(temas.value_counts().head(12).to_frame('publicaciones'))
ESTADO[4] = 'Completo' if len(sedici) > 1000 else 'No completo'
print('Estado:', ESTADO[4])

**Resultado:** SEDICI informa 191.925 registros en total y 2.618 en la Facultad de Informática; se recuperaron y analizaron **los 2.618**. La producción crece desde 2010 y tiene su pico entre 2015 y 2018; más de la mitad son **tesis** y le siguen los trabajos de **congresos**. Los temas que más se repiten (además de "Informática") son **educación**, **software** e **ingeniería de software**. **Estado: completo.**

---
## Ejercicio 5 — Indexador de documentos con interfaz gráfica
**Objetivo:** elegir el tipo de archivo, agregar un documento y buscar por texto; el índice se actualiza solo cada vez que se agrega un archivo.

**Cómo funciona:** cada documento se convierte a texto (TXT, CSV, HTML con BeautifulSoup, PDF con `pypdf`, Word con `python-docx`) y se guarda en un **índice invertido**: `palabra → {documento: cantidad de veces}`. Es procesamiento local (no hay API ni scraping).

In [ ]:
import shutil
from collections import defaultdict
import ipywidgets as widgets

CARPETA = 'indexador_documentos'
shutil.rmtree(CARPETA, ignore_errors=True)   # se arranca de cero en cada ejecución
os.makedirs(CARPETA, exist_ok=True)
TIPOS = {'Texto (.txt)': '.txt', 'PDF (.pdf)': '.pdf', 'Word (.docx)': '.docx', 'Página web (.html)': '.html', 'Planilla (.csv)': '.csv'}

def extraer_texto(ruta):
    ext = os.path.splitext(ruta)[1].lower()
    if ext in ('.txt', '.csv'):
        return open(ruta, encoding='utf-8', errors='ignore').read()
    if ext == '.html':
        return BeautifulSoup(open(ruta, encoding='utf-8', errors='ignore').read(), 'html.parser').get_text(' ')
    if ext == '.pdf':
        from pypdf import PdfReader
        return ' '.join(p.extract_text() or '' for p in PdfReader(ruta).pages)
    if ext == '.docx':
        import docx
        return ' '.join(p.text for p in docx.Document(ruta).paragraphs)
    return ''

indice = defaultdict(dict)   # palabra -> {documento: veces}
textos = {}                  # documento -> texto completo

modificado = {}             # documento -> fecha de modificación ya indexada

def indexar(ruta):
    nombre = os.path.basename(ruta)
    for docs_ in indice.values():   # si el archivo se reemplazó, se borra su versión anterior
        docs_.pop(nombre, None)
    modificado[nombre] = os.path.getmtime(ruta)
    try:
        textos[nombre] = extraer_texto(ruta)
    except Exception as error:      # p. ej. un PDF con contraseña: se avisa y no se vuelve a intentar
        print(f'No se pudo leer {nombre}: {error}')
        textos[nombre] = ''
    for palabra, veces in Counter(re.findall(r'[a-zñ0-9]{3,}', sin_tildes(textos[nombre]))).items():
        indice[palabra][nombre] = veces

def sincronizar_carpeta():
    """Indexa automáticamente cualquier archivo nuevo (o reemplazado) de la carpeta."""
    nuevos = [f for f in sorted(os.listdir(CARPETA)) if os.path.splitext(f)[1].lower() in TIPOS.values()
              and modificado.get(f) != os.path.getmtime(os.path.join(CARPETA, f))]
    for f in nuevos:
        indexar(os.path.join(CARPETA, f))
    return nuevos

def buscar(consulta):
    sincronizar_carpeta()
    terminos = re.findall(r'[a-zñ0-9]{3,}', sin_tildes(consulta))
    puntaje = Counter()
    for t in terminos:
        for doc, veces in indice.get(t, {}).items():
            puntaje[doc] += veces
    resultados = []
    for doc, pts in puntaje.most_common():
        plano = sin_tildes(textos[doc])
        pos = min([plano.find(t) for t in terminos if t in plano] or [0])
        resultados.append({'documento': doc, 'apariciones': pts, 'fragmento': ' '.join(textos[doc][max(0, pos - 80):pos + 120].split())})
    return resultados

# Para que el buscador no arranque vacío, se cargan 5 notas reales del ejercicio 3 como archivos .txt
for f in noticias.head(5).itertuples():
    with open(os.path.join(CARPETA, f'nota_{f.Index + 1}_{sin_tildes(f.diario).replace("/", "")[:10]}.txt'), 'w', encoding='utf-8') as a:
        a.write(f.titulo + '\n\n' + f.texto)
print('Documentos indexados:', sincronizar_carpeta())

In [ ]:
# Interfaz gráfica (ipywidgets). En Colab: elegir tipo -> "Agregar archivo" -> escribir una palabra -> Buscar.
tipo = widgets.Dropdown(options=list(TIPOS), description='Tipo:')
subir = widgets.FileUpload(accept='.txt', multiple=True, description='Agregar archivo')
caja = widgets.Text(placeholder='Escribí una palabra…', description='Buscar:', continuous_update=False)
boton = widgets.Button(description='Buscar', button_style='primary')
estado = widgets.HTML()
salida = widgets.Output()

def mostrar_estado(extra=''):
    estado.value = f'<b>Índice:</b> {len(textos)} documentos, {len(indice)} palabras distintas. {extra}'

def cambiar_tipo(cambio):
    subir.accept = TIPOS[tipo.value]   # el selector de archivos solo deja elegir ese tipo

def al_subir(cambio):
    valor = subir.value   # ipywidgets 7 (Colab) usa dict, ipywidgets 8 usa tupla
    archivos = [{'name': k, 'content': v['content']} for k, v in valor.items()] if isinstance(valor, dict) else list(valor)
    for a in archivos:
        if a['name'].lower().endswith(TIPOS[tipo.value]):
            with open(os.path.join(CARPETA, a['name']), 'wb') as f:
                f.write(bytes(a['content']))
    mostrar_estado('Indexado automáticamente: ' + (', '.join(sincronizar_carpeta()) or 'nada nuevo'))

def al_buscar(_):
    salida.clear_output()
    with salida:
        res = buscar(caja.value)
        print(f'{len(res)} documento(s) con "{caja.value}"')
        for r in res:
            display(HTML(f"<p><b>{html_lib.escape(r['documento'])}</b> ({r['apariciones']} apariciones)<br><i>…{html_lib.escape(r['fragmento'])}…</i></p>"))

tipo.observe(cambiar_tipo, names='value'); subir.observe(al_subir, names='value')
boton.on_click(al_buscar); caja.observe(al_buscar, names='value')   # Enter también busca
mostrar_estado()
display(widgets.VBox([widgets.HTML('<h3>Indexador de documentos</h3>'), widgets.HBox([tipo, subir]), estado, widgets.HBox([caja, boton]), salida]))

**Prueba para mostrar en clase:** la celda siguiente crea `prueba_clase.txt` con una palabra que no está en ninguna nota (**ornitorrinco**). Primero se busca y no aparece; se agrega el archivo a la carpeta y, sin reconstruir nada a mano, la búsqueda lo encuentra. En vivo se puede hacer lo mismo con el botón **Agregar archivo** (el archivo queda también en `datos/prueba_clase.txt`).

In [ ]:
print('Antes de agregar el archivo:', buscar('ornitorrinco'))
with open('datos/prueba_clase.txt', 'w', encoding='utf-8') as f:
    f.write('Documento de prueba para la clase de APIs y Web Scraping.\nLa palabra clave es ornitorrinco.')
shutil.copy('datos/prueba_clase.txt', CARPETA)   # equivale a usar el botón "Agregar archivo"
for r in buscar('ornitorrinco'):
    print('Después:', r)
print('Búsqueda de "inflación" en las notas indexadas:', [(r['documento'], r['apariciones']) for r in buscar('inflación')])
ESTADO[5] = 'Completo' if buscar('ornitorrinco') else 'No completo'
print('Estado:', ESTADO[5])

---
## Ejercicio 6 — Publicaciones de un perfil público de Facebook (sin API)
**Objetivo:** recuperar 50 publicaciones de una página pública (Infobae) y analizar su texto.

**Fuente:** `facebook.com/infobae` y el *plugin* público de páginas (**scraping**).

In [ ]:
PAGINA_FB = 'https://www.facebook.com/infobae'
r = requests.get(PAGINA_FB, headers=UA, timeout=30)
pide_login = 'login' in r.url or 'id="login_form"' in r.text or 'loginbutton' in r.text
print('Página pedida:', PAGINA_FB, '| URL final:', r.url.split('?')[0], '| pide iniciar sesión:', pide_login)
plugin = requests.get('https://www.facebook.com/plugins/page.php', headers=UA, timeout=30,
                      params={'href': PAGINA_FB, 'tabs': 'timeline', 'width': 500, 'height': 1500})
texto_plugin = BeautifulSoup(plugin.text, 'html.parser').get_text(' ', strip=True)
seguidores = re.search(r'[\d.,]+\s*(mill\.|mil)?\s*seguidores', texto_plugin)
print('Plugin público -> seguidores:', seguidores.group() if seguidores else 'no visible')
publicaciones = len(re.findall(r'userContent|data-ad-preview="message"', plugin.text + r.text))
print('Publicaciones con texto encontradas sin iniciar sesión:', publicaciones)
ESTADO[6] = 'Completo' if publicaciones >= 50 else f'No completo: Facebook exige iniciar sesión para ver publicaciones ({publicaciones} recuperadas)'
print('Estado:', ESTADO[6])

**Qué pude verificar:** la página existe y es pública (el plugin muestra nombre y cantidad de seguidores), pero Facebook redirige a la pantalla de inicio de sesión y no entrega el texto de las publicaciones a un visitante anónimo; tampoco las copias de Internet Archive tenían publicaciones. Recolectar con una cuenta personal implicaría automatizar un login, lo que viola los términos de Facebook, así que no se hizo. **Estado: no completo.**

---
## Ejercicio 7 — Publicaciones de un perfil público de X/Twitter (sin API)
**Objetivo:** recuperar al menos 50 publicaciones de **@INDECArgentina** y analizar de qué habla.

**Fuente:** Nitter (`nitter.tiekoetter.is`), una interfaz pública y alternativa que muestra los tuits de X. Como la página arma su contenido con JavaScript, se usa un navegador automático (Playwright) para hacer **scraping** y seguir el enlace "Load more" de página en página (**crawling**).

In [ ]:
NITTER = 'https://nitter.tiekoetter.is'

def leer_tuits(pagina, selector):
    tuits = []
    for item in pagina.locator(selector).all():
        try:
            fecha = item.locator('.tweet-date a').first
            tuits.append({'enlace': NITTER + fecha.get_attribute('href').split('#')[0], 'fecha': fecha.get_attribute('title'),
                          'usuario': item.locator('.username').first.inner_text(),
                          'texto': item.locator('.tweet-content').first.inner_text(),
                          'es_retweet': item.locator('.retweet-header').count() > 0})
        except Exception:
            pass
    return tuits

def esperar(pagina):
    for _ in range(40):   # Nitter a veces muestra primero una verificación anti-robots
        if pagina.locator('.timeline-item, .main-tweet').count(): return
        time.sleep(2)

def recolectar_perfil(usuario, minimo=70):
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        navegador = p.chromium.launch(headless=True); pagina = navegador.new_page(user_agent=UA['User-Agent'])
        url, tuits = f'{NITTER}/{usuario}', []
        while url and sum(not t['es_retweet'] for t in tuits) < minimo:
            pagina.goto(url, timeout=90000); esperar(pagina)
            tuits += leer_tuits(pagina, '.timeline > .timeline-item')
            mas = pagina.locator('.show-more a')
            url = f'{NITTER}/{usuario}' + mas.last.get_attribute('href') if mas.count() else None
        navegador.close()
    return pd.DataFrame(tuits).drop_duplicates('enlace')

RECOLECTAR_X = False   # True = volver a recolectar (en Colab instala Playwright, tarda ~2 minutos)
if RECOLECTAR_X:
    !pip install -q playwright && playwright install --with-deps chromium
    # Colab ya corre un "event loop"; por eso la recolección se lanza en un proceso aparte
    import multiprocessing
    with multiprocessing.get_context('fork').Pool(1) as pool:
        pool.apply(recolectar_perfil, ('INDECArgentina',)).to_csv('datos/ej7_x_indec.csv', index=False)
x = pd.read_csv('datos/ej7_x_indec.csv')
x = x[~x.es_retweet]
fechas = pd.to_datetime(x.fecha.str.replace(' · ', ' '), format='%b %d, %Y %I:%M %p UTC')
print(f'Publicaciones propias de @INDECArgentina recuperadas: {len(x)} (del {fechas.min():%d/%m/%Y} al {fechas.max():%d/%m/%Y})')
display(x[['fecha', 'texto']].head())

In [ ]:
hashtags = Counter(h.lower() for t in x.texto for h in re.findall(r'#\w+', t))
frecuentes = Counter(p for t in x.texto for p in palabras(re.sub(r'#\w+|\S+\.\S+/\S*', ' ', t)))
display(pd.DataFrame({'hashtag': [h for h, _ in hashtags.most_common(8)], 'veces': [n for _, n in hashtags.most_common(8)]}).join(
        pd.DataFrame({'palabra': [w for w, _ in frecuentes.most_common(8)], 'veces ': [n for _, n in frecuentes.most_common(8)]})))
TEMAS_INDEC = {'Precios / inflación': 'precio|ipc|inflaci|canasta', 'Actividad / industria': 'actividad|industri|construcci|emae',
               'Comercio exterior': 'export|import|intercambio', 'Empleo / salarios': 'salari|empleo|desocupa|trabajo',
               'Pobreza / ingresos': 'pobreza|ingreso|indigencia'}
temas_x = pd.Series({k: x.texto.map(sin_tildes).str.contains(v).sum() for k, v in TEMAS_INDEC.items()}).sort_values()
temas_x.plot.barh(title='Publicaciones de @INDECArgentina por tema', figsize=(7, 3)); plt.show()
ESTADO[7] = 'Completo' if len(x) >= 50 else f'No completo ({len(x)})'
print('Estado:', ESTADO[7])

**Resultado:** se recuperaron 70 publicaciones propias (sin retuits). 49 son `#DatoINDEC`: la cuenta funciona casi como un boletín de estadísticas. El tema más frecuente es **actividad / industria**, seguido de **precios / inflación**; palabras como *interanual* y *respecto* muestran que casi todo son variaciones comparadas contra otro período. **Estado: completo.**

---
## Ejercicio 8 — Respuestas a un único tuit (sin API)
**Objetivo:** tomar un tuit y recuperar al menos 50 respuestas de su hilo para analizar qué opina la gente.

**Tuit elegido:** el #DatoINDEC del 24/11/2026 sobre la actividad económica de septiembre. Misma técnica que el punto 7 (**scraping** con Playwright sobre Nitter y **crawling** por el enlace "Load more" de las respuestas).

In [ ]:
TUIT = '/INDECArgentina/status/2125303423910211839'

def recolectar_respuestas(tuit, maximo=200):
    from playwright.sync_api import sync_playwright
    with sync_playwright() as p:
        navegador = p.chromium.launch(headless=True); pagina = navegador.new_page(user_agent=UA['User-Agent'])
        url, respuestas, principal = NITTER + tuit, [], None
        while url and len(respuestas) < maximo:
            pagina.goto(url, timeout=90000); esperar(pagina)
            principal = principal or leer_tuits(pagina, '.main-tweet')
            respuestas += leer_tuits(pagina, '.replies .timeline-item')
            mas = pagina.locator('.show-more a')
            href = mas.last.get_attribute('href') if mas.count() else None
            url = (NITTER + tuit + href) if href and href.startswith('?') else None
        navegador.close()
    r = pd.DataFrame(respuestas).drop_duplicates('enlace')
    return pd.DataFrame(principal), r[~r.enlace.str.contains(tuit.split('/')[-1])]

if RECOLECTAR_X:
    with multiprocessing.get_context('fork').Pool(1) as pool:
        principal, respuestas = pool.apply(recolectar_respuestas, (TUIT,))
    principal.to_csv('datos/ej8_tuit_principal.csv', index=False); respuestas.to_csv('datos/ej8_respuestas_tuit.csv', index=False)
principal = pd.read_csv('datos/ej8_tuit_principal.csv'); respuestas = pd.read_csv('datos/ej8_respuestas_tuit.csv')
print('Tuit:', principal.texto[0].replace('\n', ' '))
print(f'Respuestas recuperadas: {len(respuestas)} de {respuestas.usuario.nunique()} usuarios distintos')
display(respuestas[['usuario', 'texto']].head(6))

In [ ]:
# Análisis simple: palabras más usadas y tono con una lista corta de palabras (no es un modelo, es un conteo transparente)
frec_resp = Counter(p for t in respuestas.texto.fillna('') for p in palabras(re.sub(r'@\w+', ' ', t)))
display(pd.DataFrame(frec_resp.most_common(12), columns=['palabra', 'veces']).T)
POSITIVAS = 'bien|buen|mejor|crec|repunt|subi|excelente|recupera|vamos|joya'
NEGATIVAS = 'cay|caida|recesion|mal|peor|mentir|miente|crisis|desastre|hambre|cierr|destru'
texto_r = respuestas.texto.fillna('').map(sin_tildes)
pos, neg = texto_r.str.contains(POSITIVAS), texto_r.str.contains(NEGATIVAS)
tono = pd.Series({'Solo palabras positivas': (pos & ~neg).sum(), 'Solo palabras negativas': (neg & ~pos).sum(),
                  'Mezcla': (pos & neg).sum(), 'Sin palabras de la lista': (~pos & ~neg).sum()})
display(tono.to_frame('respuestas').T)
ESTADO[8] = 'Completo' if len(respuestas) >= 50 else f'No completo ({len(respuestas)})'
print('Estado:', ESTADO[8])

**Resultado:** 80 respuestas de 73 usuarios distintos al tuit sobre la actividad de septiembre. La palabra más usada es **recesión**, y aparecen *Milei*, *crecimiento* y *Gobierno*: la gente discute el dato en clave política. El tono está **dividido** (cantidades parecidas de respuestas con palabras positivas y negativas), y muchas respuestas son memes o emojis que una lista de palabras no clasifica. **Estado: completo.**

---
## Ejercicio 9 — Expansión de consultas con sinónimos y un tesauro
**Objetivo:** buscar "inflación" sobre un mismo conjunto de noticias, primero tal cual y después ampliada con términos del **Tesauro de la UNESCO** y sinónimos de **WordNet en español**, y ver qué notas nuevas aparecen.

**Documentos:** las notas del punto 3 más una segunda búsqueda general de economía en los mismos diez diarios (`datos/noticias_economia.csv`), para que haya notas que hablan de precios sin usar la palabra "inflación".
**Fuentes de términos:** Tesauro UNESCO por su punto **SPARQL (API)** y WordNet (Open Multilingual Wordnet, vía NLTK).

In [ ]:
corpus = pd.concat([pd.read_csv('datos/noticias_inflacion.csv'), pd.read_csv('datos/noticias_economia.csv')]).drop_duplicates('url').reset_index(drop=True)
corpus['plano'] = (corpus.titulo + '. ' + corpus.texto).map(sin_tildes)
print('Documentos sobre los que se ejecutan las dos consultas:', len(corpus), '| diarios:', corpus.diario.nunique())

SPARQL = 'https://vocabularies.unesco.org/sparql'
pregunta = """
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?relacion ?termino WHERE {
  ?c skos:prefLabel "Inflación"@es .
  VALUES ?relacion { skos:related skos:broader skos:narrower skos:altLabel }
  ?c ?relacion ?x .
  OPTIONAL { ?x skos:prefLabel ?etq . FILTER(lang(?etq) = "es") }
  BIND(COALESCE(?etq, ?x) AS ?termino)
  FILTER(isLiteral(?termino) && lang(?termino) = "es")
}"""
try:
    r = requests.get(SPARQL, params={'query': pregunta}, headers={'Accept': 'application/sparql-results+json'}, timeout=60)
    tesauro = pd.DataFrame([{'relación': f['relacion']['value'].split('#')[1], 'término': f['termino']['value']} for f in r.json()['results']['bindings']])
    tesauro.to_csv('datos/ej9_tesauro_unesco.csv', index=False)
except Exception as e:
    print('El SPARQL de UNESCO no respondió, uso la copia guardada:', e)
    tesauro = pd.read_csv('datos/ej9_tesauro_unesco.csv')
display(tesauro)

In [ ]:
from nltk.corpus import wordnet as wn
sinonimos_aumento = sorted({l.replace('_', ' ') for s in wn.synsets('aumento', lang='spa') for l in s.lemma_names('spa')})
print('WordNet, sinónimos de "aumento":', sinonimos_aumento)

CONSULTA = 'inflación'
# Se arman frases "X de precios" con los sinónimos de WordNet que tienen sentido en economía
EXPANSION = {'Tesauro UNESCO': list(tesauro['término']),
             'WordNet (aumento → X de precios)': [f'{s} de precios' for s in ['aumento', 'alza', 'subida', 'incremento', 'suba']],
             'Siglas del dominio': ['IPC', 'índice de precios']}
TERMINOS = [CONSULTA] + [t for lista in EXPANSION.values() for t in lista]
display(pd.DataFrame([(fuente, t) for fuente, lista in EXPANSION.items() for t in lista], columns=['fuente', 'término agregado']))

def busqueda_booleana(terminos):
    """Devuelve, por documento, qué términos aparecen (búsqueda OR, sin tildes, palabra completa)."""
    encontrados = corpus.plano.apply(lambda t: [x for x in terminos if re.search(r'\b' + sin_tildes(x) + r'\b', t)])
    return encontrados[encontrados.str.len() > 0]

original = busqueda_booleana([CONSULTA])
expandida = busqueda_booleana(TERMINOS)
solo_expandida = expandida.index.difference(original.index)
print(f'Consulta original  "{CONSULTA}": {len(original)} documentos')
print(f'Consulta expandida ({len(TERMINOS)} términos): {len(expandida)} documentos  ->  {len(solo_expandida)} nuevos')

In [ ]:
comparacion = pd.DataFrame({'original': corpus.loc[original.index].diario.value_counts(),
                            'expandida': corpus.loc[expandida.index].diario.value_counts()}).fillna(0).astype(int)
comparacion['ganancia'] = comparacion.expandida - comparacion.original
display(comparacion.sort_values('ganancia', ascending=False).T)
aportes = Counter(t for i in solo_expandida for t in expandida[i])
print('Términos que trajeron las notas nuevas:', dict(aportes.most_common()))

def resaltar(texto, termino, ancho=160):
    plano = sin_tildes(texto); m = re.search(r'\b' + sin_tildes(termino) + r'\b', plano)
    a, b = max(0, m.start() - ancho), m.end() + ancho
    e = html_lib.escape
    return '…' + e(texto[a:m.start()]) + '<mark><b>' + e(texto[m.start():m.end()]) + '</b></mark>' + e(texto[m.end():b]) + '…'

display(HTML('<h4>Ejemplos concretos: notas que solo encuentra la consulta expandida</h4>' + ''.join(
    f"<p><b>{html_lib.escape(corpus.diario[i])}</b> — {link(corpus.url[i], corpus.titulo[i])}<br>Coincide por: <code>{', '.join(expandida[i])}</code><br>"
    f"{resaltar(corpus.titulo[i] + '. ' + corpus.texto[i], expandida[i][0])}</p>" for i in solo_expandida[:4])))
ESTADO[9] = 'Completo' if len(solo_expandida) > 0 else 'No completo'
print('Estado:', ESTADO[9])

**Qué cambió:** sobre los mismos 203 documentos, "inflación" sola recupera 150 notas y la consulta expandida 175 (**+25**). Ejemplo: la nota de TN sobre el aumento del 2,2 % en prestaciones por discapacidad nunca dice "inflación" pero habla del *Índice de Precios al Consumidor (IPC)*; solo la encuentra la consulta expandida. También se ve el costo: el término del tesauro "Precio" trae notas de recesión que mencionan el *precio del gas*, que son menos pertinentes. Expandir mejora la recuperación (encuentra más) pero puede bajar la precisión. **Estado: completo.**

---
## Ejercicio 10 — TF-IDF y LSI/LSA sobre 100 noticias
**Objetivo:** representar 100 o más noticias distintas sobre inflación con TF-IDF y con LSA, ejecutar la misma consulta y comparar los primeros resultados.

**Documentos:** las notas descargadas en el punto 3 (`datos/noticias_inflacion.csv`).
- **TF-IDF:** cada nota es un vector de palabras; pesan más las palabras frecuentes en la nota y raras en el resto. Solo encuentra notas que comparten palabras con la consulta.
- **LSA:** descompone la matriz TF-IDF (SVD) en 20 "temas latentes"; dos textos se parecen si usan palabras que suelen aparecer juntas, aunque no sean las mismas.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

docs = noticias.drop_duplicates('url').drop_duplicates('texto').reset_index(drop=True)
print('Noticias distintas:', len(docs))
vectorizador = TfidfVectorizer(preprocessor=sin_tildes, stop_words=sorted(VACIAS), min_df=2, max_df=0.8)
X_tfidf = vectorizador.fit_transform(docs.titulo + '. ' + docs.texto)
lsa = make_pipeline(TruncatedSVD(n_components=20, random_state=0), Normalizer(copy=False))
X_lsa = lsa.fit_transform(X_tfidf)
print('Matriz TF-IDF:', X_tfidf.shape, '(notas x palabras) | Matriz LSA:', X_lsa.shape, '(notas x temas)')
print(f"Varianza explicada por los 20 temas: {lsa[0].explained_variance_ratio_.sum():.0%}")

In [ ]:
CONSULTA_10 = 'salarios y poder adquisitivo'
q_tfidf = vectorizador.transform([CONSULTA_10])
q_lsa = lsa.transform(q_tfidf)
docs['score_tfidf'] = (X_tfidf @ q_tfidf.T).toarray().ravel()
docs['score_lsa'] = X_lsa @ q_lsa.ravel()
N = 10
top_t = docs.nlargest(N, 'score_tfidf'); top_l = docs.nlargest(N, 'score_lsa')
tabla10 = pd.DataFrame({'TF-IDF': [f'{f.diario}: {f.titulo[:70]}' for f in top_t.itertuples()], 'puntaje TF-IDF': top_t.score_tfidf.round(3).values,
                        'LSA': [f'{f.diario}: {f.titulo[:70]}' for f in top_l.itertuples()], 'puntaje LSA': top_l.score_lsa.round(3).values},
                       index=range(1, N + 1))
en_ambos = set(top_t.index) & set(top_l.index)
print(f'Consulta: "{CONSULTA_10}" | notas que aparecen en el top {N} de los dos métodos: {len(en_ambos)}')
display(tabla10.style.apply(lambda col: ['background:#d4edda' if i in en_ambos else '' for i in (top_t.index if col.name.endswith('TF-IDF') else top_l.index)]))

In [ ]:
# ¿Por qué difieren? Para las notas que solo trae LSA, contamos cuántas palabras de la consulta contienen realmente
claves = [p for p in palabras(CONSULTA_10)]
solo_lsa = [i for i in top_l.index if i not in top_t.index]
for i in solo_lsa[:4]:
    t = sin_tildes(docs.texto[i])
    print(f"LSA puesto {list(top_l.index).index(i) + 1}: {docs.titulo[i][:80]}")
    print('   palabras de la consulta presentes:', {c: t.count(c) for c in claves},
          '| puesto en TF-IDF:', int((docs.score_tfidf > docs.score_tfidf[i]).sum()) + 1)
ESTADO[10] = 'Completo' if len(docs) >= 100 else f'No completo ({len(docs)} noticias)'
print('Estado:', ESTADO[10])

**Coincidencia y diferencia:** 5 de los 10 primeros resultados son iguales (en verde), por ejemplo la nota de Perfil sobre cuánto aumentarían los salarios en 2027, que tiene las palabras de la consulta y LSA la confirma por tema. La diferencia: TF-IDF compara palabras exactas, entonces la nota de Infobae sobre el **salario mínimo** (dice *salario*, no *salarios*) queda en el puesto 15, y la de Clarín *"menos Wall Street y más Conurbano"*, que no contiene ninguna palabra de la consulta, queda en el 65. LSA las sube al 5.º y 8.º lugar porque usan el mismo vocabulario de fondo (ingresos, consumo, trabajo): busca por tema, no por palabra exacta. **Estado: completo** (130 noticias distintas).

---
## Estado de los diez ejercicios
Cada estado se calcula con las cantidades obtenidas en esta ejecución (no está escrito a mano).

In [ ]:
NOMBRES = {1: 'Mercado Libre', 2: 'Avisos clasificados', 3: 'Noticias en 10 diarios', 4: 'Repositorio SEDICI', 5: 'Indexador',
           6: 'Facebook', 7: 'X: perfil público', 8: 'X: respuestas a un tuit', 9: 'Expansión de consultas', 10: 'TF-IDF y LSA'}
METODO = {1: 'Scraping + API (bloqueados)', 2: 'Scraping', 3: 'API (RSS) + crawling + scraping', 4: 'API OAI-PMH + crawling',
          5: 'Procesamiento local + interfaz', 6: 'Scraping (bloqueado)', 7: 'Scraping + crawling (Nitter)', 8: 'Scraping + crawling (Nitter)',
          9: 'API SPARQL (UNESCO) + WordNet', 10: 'TF-IDF + SVD (scikit-learn)'}
display(pd.DataFrame({'ejercicio': NOMBRES, 'método': METODO, 'estado': ESTADO}).sort_index())

---
## Guion de 2 minutos para la exposición
Antes de la clase: *Entorno de ejecución → Ejecutar todas* (tarda unos 3 minutos) y dejar el cuaderno abierto en el punto 3.

| Tiempo | Celda | Qué señalar | Qué decir |
|---|---|---|---|
| 0:00–0:20 | Tabla de estado (final) | La columna *método* y los dos "No completo" | "Hice los diez puntos; en cada uno indico si usé API, crawling o scraping. Mercado Libre y Facebook bloquean la recolección y lo dejé documentado, sin inventar datos." |
| 0:20–0:45 | Ej. 3, celda de búsqueda (volver a ejecutarla) | El gráfico de barras con los 10 diarios | "Consulto el RSS de Google News filtrado por cada diario. Ahora mismo, en vivo, encuentra noticias de inflación en los diez." |
| 0:45–1:15 | Ej. 9, celda de la consulta expandida | *150 documentos → 175* y la nota de TN marcada en amarillo | "Busco 'inflación' y después agrego términos del Tesauro de la UNESCO y sinónimos de WordNet. Sobre las mismas notas aparecen 25 nuevas: esta habla del IPC sin decir 'inflación'." |
| 1:15–1:40 | Ej. 10, tabla TF-IDF vs LSA | Las filas verdes y la nota de *salario mínimo* | "TF-IDF busca las palabras exactas; LSA busca por tema. Por eso LSA encuentra 'salario mínimo' aunque yo escribí 'salarios'." |
| 1:40–2:00 | Ej. 5, indexador | Escribir **ornitorrinco** y apretar *Buscar* (o subir `datos/prueba_clase.txt`) | "Al agregar el archivo el índice se actualiza solo y el buscador lo encuentra." |

Para cambiar la búsqueda en vivo: modificar `TEMA` (punto 3), `CONSULTA` (punto 9) o `CONSULTA_10` (punto 10) y volver a ejecutar esa celda y las siguientes del mismo punto.